# Titanic v2 — Улучшенная LinearRegression

Новые признаки по сравнению с v1:

| Признак | Идея |
|---------|------|
| **AgeBin** | Дети выживали чаще — нелинейная зависимость от возраста |
| **FareBin** | Цена билета нелинейно связана с классом и шансами |
| **FamilySizeGroup** | Маленькая семья (2-4) выживала лучше одиночек и больших семей |
| **Sex_Pclass** | Взаимодействие пола и класса: женщина 1-го ≠ женщина 3-го |

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score

In [ ]:
# Загружаем данные
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

# Сохраняем PassengerId из test до того, как удалим колонку
test_ids = test["PassengerId"].copy()

print("train:", train.shape)
print("test: ", test.shape)

In [ ]:
# Единая функция предобработки для train и test.
# median_age и fare_bins вычисляются на train и передаются в test — нет утечки данных.
def preprocess(df, median_age=None, fare_bins=None):
    df = df.copy()

    # HasCabin: наличие каюты косвенно означает 1-й/2-й класс → ближе к шлюпкам
    df["HasCabin"] = df["Cabin"].notna().astype(int)
    df.drop(columns=["Cabin"], inplace=True)

    # Title: Mr/Miss/Mrs/Master кодирует пол и возраст одновременно
    # expand=False возвращает Series, а не DataFrame — иначе .str.strip() упадёт
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    df["Title"] = df["Title"].map({"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4}).fillna(5).astype(int)

    # Sex: women and children first — самый сильный признак
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

    # Embarked: заполняем пропуски самым частым портом S, кодируем числом
    df["Embarked"] = df["Embarked"].fillna("S").map({"S": 0, "C": 1, "Q": 2})

    # Fare: заполняем единственный пропуск в test медианой
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())

    # Age: заполняем медианой по группе Pclass + Sex (только из train)
    if median_age is None:
        median_age = df.groupby(["Pclass", "Sex"])["Age"].median().to_dict()
    df["Age"] = df.apply(
        lambda row: median_age.get((row["Pclass"], row["Sex"]), df["Age"].median())
        if pd.isna(row["Age"]) else row["Age"],
        axis=1
    )

    # FamilySize, IsAlone
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"]    = (df["FamilySize"] == 1).astype(int)

    # FamilySizeGroup: 0=один, 1=маленькая (2-4), 2=большая (5+)
    df["FamilySizeGroup"] = df["FamilySize"].apply(
        lambda x: 0 if x == 1 else (1 if x <= 4 else 2)
    )

    # AgeBin: 0=ребёнок (0-12), 1=подросток (12-18), 2=взрослый (18-60), 3=пожилой (60+)
    df["AgeBin"] = pd.cut(df["Age"], bins=[0, 12, 18, 60, 100],
                          labels=[0, 1, 2, 3]).astype(int)

    # FareBin: 4 квантильные корзины — границы считаем на train, применяем к test
    if fare_bins is None:
        df["FareBin"], fare_bins = pd.qcut(df["Fare"], q=4, labels=[0, 1, 2, 3],
                                           retbins=True, duplicates="drop")
    else:
        df["FareBin"] = pd.cut(df["Fare"].clip(upper=fare_bins[-1]),
                               bins=fare_bins, labels=False, include_lowest=True)
    df["FareBin"] = df["FareBin"].fillna(0).astype(int)

    # Sex_Pclass: взаимодействие пола и класса
    # Женщина 1-го класса (1×1=1) vs мужчина 3-го (0×3=0)
    df["Sex_Pclass"] = df["Sex"] * df["Pclass"]

    # Удаляем лишние колонки
    drop_cols = ["Name", "Ticket", "SibSp", "Parch"]
    if "PassengerId" in df.columns:
        drop_cols.append("PassengerId")
    df.drop(columns=drop_cols, inplace=True)

    return df, median_age, fare_bins


# Применяем: сначала train (вычисляем параметры), потом test (передаём параметры)
train, median_age, fare_bins = preprocess(train)
test,  _,          _         = preprocess(test, median_age=median_age, fare_bins=fare_bins)

print("train:", train.shape)
print("Признаки:", list(train.columns))
print("\nПропуски:\n", train.isnull().sum())

In [ ]:
features = ["Pclass", "Sex", "Age", "AgeBin", "Fare", "FareBin",
            "Embarked", "FamilySize", "IsAlone", "FamilySizeGroup",
            "HasCabin", "Title", "Sex_Pclass"]

X_train = train[features]
y_train = train["Survived"]
X_test  = test[features]

# Обучаем модель
model = LinearRegression()
model.fit(X_train, y_train)
train_preds = model.predict(X_train)

# Подбор лучшего порога от 0.3 до 0.7
best_thr, best_acc = 0.5, 0.0
for thr in np.arange(0.3, 0.71, 0.01):
    acc = accuracy_score(y_train, (train_preds >= thr).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_thr = round(thr, 2)

print(f"Лучший порог: {best_thr}  |  Accuracy на train: {best_acc:.4f} ({best_acc*100:.2f}%)")

# Предсказания на test
test_preds = (model.predict(X_test) >= best_thr).astype(int)

# Сохраняем submission_v2.csv
submission = pd.DataFrame({"PassengerId": test_ids, "Survived": test_preds})
submission.to_csv("submission_v2.csv", index=False)

print("\nsubmission_v2.csv сохранён — загружай на Kaggle!")
print(submission.head())
print("\nРаспределение:", submission["Survived"].value_counts().to_dict())

In [ ]:
# Веса модели — смотрим, какой признак влияет сильнее всего
coef_df = pd.DataFrame({
    "Признак": features,
    "Вес":     model.coef_.round(4)
}).sort_values("Вес", ascending=False)

print("Веса признаков (чем больше по модулю — тем сильнее влияние):")
print(coef_df.to_string(index=False))